This example demonstrates how Bayesian optimization can be applied to optimize multi-material properties. First, we train an SVM model using experimental data. This pre-trained model acts as a surrogate for a real experimental simulator. During the optimization process, when new data points are recommended, the simulator evaluates their properties, allowing us to bypass real experiments

In [6]:
import pandas as pd
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
import joblib

# Load the dataset
data = pd.read_csv('data.csv')

# Extract features and targets
X = data.iloc[:, :4]  # First four columns as features
y = data.iloc[:, -2:]  # Last two columns as targets

# Split the dataset into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Since SVR supports only single target, loop through each target column
y_train_1, y_train_2 = y_train.iloc[:, 0], y_train.iloc[:, 1]
y_test_1, y_test_2 = y_test.iloc[:, 0], y_test.iloc[:, 1]

# Hyperparameter tuning for target 1
param_grid = {
    'kernel': ['rbf'],
    'C': [0.1, 0.3, 0.5, 1, 3, 5, 7, 10, 100, 1000, 10000],
    'epsilon': [0.001, 0.01, 0.1, 0.3, 0.5, 0.7, 1],
    'gamma': ['scale', 'auto']
}
grid_search_1 = GridSearchCV(SVR(), param_grid, scoring='neg_mean_squared_error', cv=5)
grid_search_1.fit(X_train, y_train_1)
svr_model_1 = grid_search_1.best_estimator_

# Hyperparameter tuning for target 2
grid_search_2 = GridSearchCV(SVR(), param_grid, scoring='neg_mean_squared_error', cv=5)
grid_search_2.fit(X_train, y_train_2)
svr_model_2 = grid_search_2.best_estimator_

# Train the models
svr_model_1.fit(X_train, y_train_1)
svr_model_2.fit(X_train, y_train_2)

# Predict on the test set
y_pred_1 = svr_model_1.predict(X_test)
y_pred_2 = svr_model_2.predict(X_test)

# Calculate the performance (Mean Squared Error) for each target
mse_1 = mean_squared_error(y_test_1, y_pred_1)
mse_2 = mean_squared_error(y_test_2, y_pred_2)

print(f'Mean Squared Error for target 1 on test set: {mse_1}')
print(f'Mean Squared Error for target 2 on test set: {mse_2}')

# Save the models and parameters
joblib.dump(svr_model_1, 'svm_model_target1.joblib')
joblib.dump(svr_model_2, 'svm_model_target2.joblib')

# Save the models' hyperparameters
with open('svm_model_params.txt', 'w') as f:
    f.write('Target 1:\n')
    f.write(str(svr_model_1.get_params()))
    f.write('\n\nTarget 2:\n')
    f.write(str(svr_model_2.get_params()))


Mean Squared Error for target 1 on test set: 7.08819692347383
Mean Squared Error for target 2 on test set: 4.878841601612547


In [7]:
# Load the trained models
svr_model_1 = joblib.load('svm_model_target1.joblib')
svr_model_2 = joblib.load('svm_model_target2.joblib')

# Input new features (replace with actual feature values)
new_features = [[92, 1, 2, 5]]  # Replace with actual feature values

# Make predictions for both targets
prediction_1 = svr_model_1.predict(new_features)
prediction_2 = svr_model_2.predict(new_features)

# Output the predicted values
print(f'Predicted value for target 1: {prediction_1[0]}')
print(f'Predicted value for target 2: {prediction_2[0]}')

Predicted value for target 1: 86.12726441454102
Predicted value for target 2: 17.13981224142781


C:\Users\binja\anaconda3\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but SVR was fitted with feature names
  warnings.warn(
C:\Users\binja\anaconda3\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but SVR was fitted with feature names
  warnings.warn(
